# Data Cleaning: Building the Movie Database

**Goal**: Merge three data sources into a single, clean dataset of movies
with `title`, `director`, `genre`, `plot`, and `popularity` columns.

## Data Sources

| Source | Movies | Has Plot | Has Director | Join Key |
|--------|--------|----------|-------------|----------|
| Our hand-written dataset | 196 | Yes | Yes | -- |
| CMU Movie Summary Corpus | 42,306 | Yes | **No** | Wikipedia ID |
| HuggingFace MovieSum | 1,800 | Yes | **No** | IMDB ID |
| IMDB datasets | 668K+ | No | **Yes** | tconst |
| IMDB ratings | 1.4M+ | No | No (numVotes) | tconst |

## Strategy
1. Build a director lookup from IMDB (title → director)
2. Join MovieSum to IMDB via `imdb_id`
3. Join CMU to IMDB via `title + year` matching
4. Merge all three, deduplicate, quality-check
5. Add popularity scores from IMDB numVotes (log-scaled, 0–1)
6. Export

---
## Cell 1: Load Raw Data

In [ ]:
import pandas as pd
import json
import os

# ── Our existing hand-written dataset ──
original = pd.read_csv('movies_dataset.csv')
original['source'] = 'hand_written'
print(f'Original dataset: {len(original)} movies, {original["director"].nunique()} directors')

# ── CMU Movie Summary Corpus ──
print('\nLoading CMU corpus...')
cmu_plots = pd.read_csv(
    'MovieSummaries/plot_summaries.txt', sep='\t',
    header=None, names=['wiki_id', 'plot']
)
cmu_meta_cols = ['wiki_id', 'freebase_id', 'title', 'release_date', 'revenue',
                 'runtime', 'languages', 'countries', 'genres_json']
cmu_meta = pd.read_csv(
    'MovieSummaries/movie.metadata.tsv', sep='\t',
    header=None, names=cmu_meta_cols
)
cmu = cmu_meta.merge(cmu_plots, on='wiki_id', how='inner')
print(f'CMU: {len(cmu_meta):,} metadata + {len(cmu_plots):,} plots → {len(cmu):,} joined')

# ── HuggingFace MovieSum (pre-extracted lightweight version) ──
if os.path.exists('moviesum_light.csv'):
    moviesum = pd.read_csv('moviesum_light.csv')
    print(f'MovieSum: {len(moviesum):,} movies (loaded from cache)')
else:
    from datasets import load_dataset
    print('Streaming MovieSum from HuggingFace...')
    ds = load_dataset('rohitsaxena/MovieSum', split='train', streaming=True)
    rows = [{'movie_name': r['movie_name'], 'imdb_id': r['imdb_id'], 'summary': r['summary']} for r in ds]
    moviesum = pd.DataFrame(rows)
    moviesum.to_csv('moviesum_light.csv', index=False)
    print(f'MovieSum: {len(moviesum):,} movies (streamed and cached)')

# ── IMDB datasets ──
print('\nLoading IMDB datasets (this takes ~30s)...')
imdb_basics = pd.read_csv(
    'imdb_title_basics.tsv.gz', sep='\t', low_memory=False,
    na_values='\\N',
    usecols=['tconst', 'titleType', 'primaryTitle', 'startYear', 'genres']
)
imdb_crew = pd.read_csv('imdb_title_crew.tsv.gz', sep='\t', na_values='\\N')
imdb_names = pd.read_csv(
    'imdb_name_basics.tsv.gz', sep='\t', low_memory=False,
    na_values='\\N', usecols=['nconst', 'primaryName']
)
print(f'IMDB: {len(imdb_basics):,} titles, {len(imdb_crew):,} crew, {len(imdb_names):,} names')

---
## Cell 2: Build IMDB Director Lookup

Join `title.basics` → `title.crew` → `name.basics` to get:
`tconst → (title, year, genres, director_name)`

In [ ]:
# Filter to movies only (drop shorts, TV series, video games, etc.)
imdb_movies = imdb_basics[imdb_basics['titleType'] == 'movie'].copy()
print(f'IMDB movies (filtered): {len(imdb_movies):,}')

# Join with crew to get director ID(s)
imdb_movies = imdb_movies.merge(imdb_crew[['tconst', 'directors']], on='tconst', how='left')

# Take first director for multi-director films (e.g. Coen Brothers)
imdb_movies['director_id'] = imdb_movies['directors'].str.split(',').str[0]

# Join with names to get director name
imdb_movies = imdb_movies.merge(
    imdb_names, left_on='director_id', right_on='nconst', how='left'
)
imdb_movies = imdb_movies.rename(columns={
    'primaryName': 'director',
    'primaryTitle': 'title',
    'startYear': 'year'
})

# Drop entries without a director
imdb_movies = imdb_movies.dropna(subset=['director'])

# Extract first genre
imdb_movies['genre'] = imdb_movies['genres'].str.split(',').str[0]

# Clean up
imdb_lookup = imdb_movies[['tconst', 'title', 'year', 'director', 'genre']].copy()
imdb_lookup['year'] = imdb_lookup['year'].astype('Int64')  # nullable int

# Create a title+year key for CMU matching
imdb_lookup['match_key'] = (imdb_lookup['title'].str.lower().str.strip() + '_' + 
                            imdb_lookup['year'].astype(str))

print(f'IMDB director lookup: {len(imdb_lookup):,} movies, {imdb_lookup["director"].nunique():,} unique directors')
print(f'\nSample:')
imdb_lookup[['title', 'year', 'director', 'genre']].head(5)

---
## Cell 3: Process MovieSum

Join MovieSum → IMDB via `imdb_id = tconst` to get directors and genres.

In [ ]:
# Clean movie names (format is "Title_Year")
moviesum['clean_name'] = moviesum['movie_name'].str.rsplit('_', n=1).str[0].str.replace('_', ' ')

# Join with IMDB lookup via imdb_id
ms_joined = moviesum.merge(
    imdb_lookup[['tconst', 'director', 'genre']],
    left_on='imdb_id', right_on='tconst',
    how='inner'
)

# Build final MovieSum dataframe
ms_final = pd.DataFrame({
    'title': ms_joined['clean_name'],
    'director': ms_joined['director'],
    'genre': ms_joined['genre'],
    'plot': ms_joined['summary'],
    'source': 'moviesum'
})

print(f'MovieSum: {len(moviesum)} raw → {len(ms_final)} with directors')
print(f'Top directors in MovieSum:')
print(ms_final['director'].value_counts().head(10).to_string())

---
## Cell 4: Process CMU Corpus

Join CMU → IMDB via exact `title + year` matching to get directors.

In [ ]:
# Parse genre from Freebase JSON
def parse_freebase_genre(genres_json):
    """Extract the first genre name from CMU's Freebase JSON format."""
    try:
        d = json.loads(genres_json)
        return list(d.values())[0] if d else None
    except:
        return None

cmu['genre'] = cmu['genres_json'].apply(parse_freebase_genre)

# Extract year from release date
cmu['year'] = pd.to_datetime(cmu['release_date'], errors='coerce').dt.year.astype('Int64')

# Create match key for IMDB join
cmu['match_key'] = (cmu['title'].str.lower().str.strip() + '_' +
                    cmu['year'].astype(str))

# Join with IMDB lookup on title+year
cmu_joined = cmu.merge(
    imdb_lookup[['match_key', 'director']].drop_duplicates(subset='match_key'),
    on='match_key',
    how='inner'
)

# Build final CMU dataframe
cmu_final = pd.DataFrame({
    'title': cmu_joined['title'],
    'director': cmu_joined['director'],
    'genre': cmu_joined['genre'],
    'plot': cmu_joined['plot'],
    'source': 'cmu'
})

print(f'CMU: {len(cmu)} with plots → {len(cmu_joined)} matched to IMDB directors')
print(f'\nTop directors in CMU:')
print(cmu_final['director'].value_counts().head(10).to_string())

---
## Cell 5: Merge All Sources

Combine all three datasets, deduplicate, and apply quality filters.

In [ ]:
# Standardize original dataset columns
orig = original[['title', 'director', 'genre', 'plot', 'source']].copy()

print('Merging sources...')
print(f'  Hand-written: {len(orig)}')
print(f'  MovieSum:     {len(ms_final)}')
print(f'  CMU:          {len(cmu_final)}')

# Concatenate (order matters: first source wins on dedup)
merged = pd.concat([orig, ms_final, cmu_final], ignore_index=True)
print(f'  Combined:     {len(merged)}')

# ── Deduplication ──
# Create a dedup key from lowercase title + director
merged['dedup_key'] = (merged['title'].str.lower().str.strip() + '|' +
                       merged['director'].str.lower().str.strip())

# Keep first occurrence (hand_written > moviesum > cmu)
merged = merged.drop_duplicates(subset='dedup_key', keep='first')
print(f'  After dedup:  {len(merged)}')

# ── Quality filters ──
merged['plot_len'] = merged['plot'].str.len()

# Drop plots that are too short (< 50 chars) or too long (> 5000 chars)
before = len(merged)
merged = merged[(merged['plot_len'] >= 50) & (merged['plot_len'] <= 5000)]
print(f'  After length filter (50-5000 chars): {len(merged)} (dropped {before - len(merged)})')

# Drop any remaining nulls
before = len(merged)
merged = merged.dropna(subset=['title', 'director', 'plot'])
print(f'  After null drop: {len(merged)} (dropped {before - len(merged)})')

# Clean up temp columns
merged = merged.drop(columns=['dedup_key', 'plot_len'])

print(f'\n=== FINAL DATASET ===')
print(f'Total movies:    {len(merged):,}')
print(f'Unique directors: {merged["director"].nunique():,}')
print(f'Unique genres:    {merged["genre"].nunique()}')
print(f'\nSource breakdown:')
print(merged['source'].value_counts().to_string())

---
## Cell 5b: Add Popularity Scores (IMDB numVotes)

When a user's plot matches two movies equally, we want the **famous** one to surface.
IMDB `numVotes` is a strong proxy for cultural prominence.

**Steps:**
1. Load `title.ratings.tsv.gz` (has `tconst`, `averageRating`, `numVotes`)
2. Join to `imdb_lookup` (from Cell 2) to get numVotes per movie
3. Match to our merged dataset via `title + director`
4. Log-scale and normalize to 0–1
5. Unmatched movies get `median × 0.5` as a conservative default

In [ ]:
import numpy as np

# ── Load IMDB ratings ──
imdb_ratings = pd.read_csv('title.ratings.tsv.gz', sep='\t')
print(f'IMDB ratings: {len(imdb_ratings):,} titles')

# ── Join ratings to our IMDB lookup (from Cell 2) to get numVotes per tconst ──
imdb_with_votes = imdb_lookup.merge(imdb_ratings[['tconst', 'numVotes']], on='tconst', how='left')

# ── Create a title+director key for matching to our merged dataset ──
imdb_with_votes['pop_key'] = (
    imdb_with_votes['title'].str.lower().str.strip() + '|' +
    imdb_with_votes['director'].str.lower().str.strip()
)

# Deduplicate: if multiple IMDB entries match, keep the one with most votes
imdb_pop = (imdb_with_votes
    .dropna(subset=['numVotes'])
    .sort_values('numVotes', ascending=False)
    .drop_duplicates(subset='pop_key', keep='first')
    [['pop_key', 'numVotes']]
)

# ── Join to merged dataset ──
merged['pop_key'] = (
    merged['title'].str.lower().str.strip() + '|' +
    merged['director'].str.lower().str.strip()
)
merged = merged.merge(imdb_pop, on='pop_key', how='left')

matched = merged['numVotes'].notna().sum()
print(f'Matched {matched:,}/{len(merged):,} movies to IMDB ratings ({matched/len(merged):.1%})')

# ── Log-scale and normalize to 0-1 ──
# Log scale compresses the huge range (1 vote to 2M+ votes)
log_votes = np.log1p(merged['numVotes'].fillna(0))
max_log = log_votes.max()
merged['popularity'] = log_votes / max_log

# Unmatched movies get a conservative default: median * 0.5
median_pop = merged.loc[merged['numVotes'].notna(), 'popularity'].median()
default_pop = median_pop * 0.5
merged.loc[merged['numVotes'].isna(), 'popularity'] = default_pop

# Clean up temp columns
merged = merged.drop(columns=['pop_key', 'numVotes'])

print(f'Popularity range: {merged["popularity"].min():.4f} - {merged["popularity"].max():.4f}')
print(f'Median (matched): {median_pop:.4f}')
print(f'Default (unmatched): {default_pop:.4f}')
print(f'\nTop 10 most popular movies:')
print(merged.nlargest(10, 'popularity')[['title', 'director', 'popularity']].to_string(index=False))

---
## Cell 6: Quality Checks

In [ ]:
print('=== QUALITY CHECKS ===')
print()

# 1. Null check
nulls = merged[['title', 'director', 'genre', 'plot', 'popularity']].isnull().sum()
print('Null values per column:')
print(nulls.to_string())
print()

# 2. Plot length distribution
plot_lens = merged['plot'].str.len()
print(f'Plot length: min={plot_lens.min()}, median={plot_lens.median():.0f}, '
      f'mean={plot_lens.mean():.0f}, max={plot_lens.max()}')
print()

# 3. Popularity distribution
print(f'Popularity: min={merged["popularity"].min():.4f}, '
      f'median={merged["popularity"].median():.4f}, '
      f'mean={merged["popularity"].mean():.4f}, '
      f'max={merged["popularity"].max():.4f}')
print()

# 4. Top 20 directors by movie count
print('Top 20 directors:')
print(merged['director'].value_counts().head(20).to_string())
print()

# 5. Top 15 genres
print('Top 15 genres:')
print(merged['genre'].value_counts().head(15).to_string())
print()

# 6. Are our original 16 directors still represented?
original_directors = set(original['director'].unique())
found = original_directors & set(merged['director'].unique())
missing = original_directors - found
print(f'Original 16 directors: {len(found)} found, {len(missing)} missing')
if missing:
    print(f'  Missing: {missing}')
print()

# 7. Random sample for visual inspection
print('=== RANDOM SAMPLE (5 entries) ===')
sample = merged.sample(5, random_state=42)
for _, row in sample.iterrows():
    print(f"\n  Title:      {row['title']}")
    print(f"  Director:   {row['director']}")
    print(f"  Genre:      {row['genre']}")
    print(f"  Popularity: {row['popularity']:.4f}")
    print(f"  Plot:       {row['plot'][:120]}...")
    print(f"  Source:     {row['source']}")

---
## Cell 7: Export

Save the final dataset as `movies_dataset.csv` with columns:
`title`, `director`, `genre`, `plot`, `popularity`.

In [ ]:
# Keep only the columns the pipeline needs
export = merged[['title', 'director', 'genre', 'plot', 'popularity']].copy()

# Backup the old dataset
if os.path.exists('movies_dataset.csv'):
    old = pd.read_csv('movies_dataset.csv')
    old.to_csv('movies_dataset_v1_backup.csv', index=False)
    print(f'Backed up old dataset ({len(old)} rows) → movies_dataset_v1_backup.csv')

# Save new dataset
export.to_csv('movies_dataset.csv', index=False)

print(f'\n=== EXPORT COMPLETE ===')
print(f'Old dataset: {len(old) if "old" in dir() else "N/A"} movies')
print(f'New dataset: {len(export):,} movies')
print(f'Saved to:    movies_dataset.csv')
print(f'File size:   {os.path.getsize("movies_dataset.csv") / 1024 / 1024:.1f} MB')
print(f'\nColumns: {list(export.columns)}')
print(f'Directors: {export["director"].nunique():,}')
print(f'Genres:    {export["genre"].nunique()}')

---
## Cell 8: Verify Pipeline Still Works

Quick sanity check: run the SBERT+TF-IDF evaluation against the new dataset.

In [ ]:
from pipeline import detect_plagiarism
from evaluation import EVAL_CASES
from schema import PLAGIARISM_THRESHOLD

df = pd.read_csv('movies_dataset.csv')
print(f'Testing pipeline with new dataset ({len(df):,} movies)...')
print(f'Threshold: {PLAGIARISM_THRESHOLD}\n')

correct = 0
total = len(EVAL_CASES)

for case in EVAL_CASES:
    result = detect_plagiarism(case['user_plot'], df)
    predicted = result['detected_plagiarism']
    expected = case['expected_plagiarism']
    match = predicted == expected
    correct += match
    symbol = 'PASS' if match else 'FAIL'
    print(f"  [{symbol}] {case['id']}: score={result['similarity_score']:.4f} "
          f"match=\"{result['matched_movie']}\" "
          f"pred={'PLAG' if predicted else 'ORIG'} "
          f"exp={'PLAG' if expected else 'ORIG'}")

print(f'\nAccuracy: {correct}/{total} ({correct/total:.1%})')
if correct == total:
    print('All eval cases still pass with the new dataset!')
else:
    print('WARNING: Some eval cases failed. Threshold may need recalibration.')